# Python `os`, `pathlib`, and `sys` — A Detailed Tutorial

This notebook covers three of Python's most commonly used standard library modules for interacting with the file system and the runtime environment:

- **`os`** — operating system interfaces: processes, environment variables, low-level file/path operations
- **`pathlib`** — modern, object-oriented filesystem paths (Python 3.4+)
- **`sys`** — interpreter-level info and control: command-line args, Python path, stdio, runtime internals

Each section includes explanations and runnable code cells. Run the cells top to bottom — some later cells depend on files/directories created earlier in the notebook.


In [1]:
import os
import sys
from pathlib import Path

print("Python version:", sys.version)
print("Current working directory:", os.getcwd())


Python version: 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
Current working directory: /home/claude


## 0. Setup — a sandbox directory

We'll create a temporary sandbox folder to safely experiment with file/directory operations without touching real project files.


In [2]:
import tempfile

# Create a temp sandbox directory for this notebook's experiments
SANDBOX = Path(tempfile.mkdtemp(prefix="ospathlib_tutorial_"))
print("Sandbox created at:", SANDBOX)


Sandbox created at: /tmp/ospathlib_tutorial_b_lut3an


---
# Part 1: The `os` Module

The `os` module gives you a portable way to use operating-system-dependent functionality: environment variables, process management, and low-level (string-based) path operations via `os.path`.


### 1.1 Current directory, environment variables, and system info

In [3]:
# Current working directory
print("cwd:", os.getcwd())

# Change directory (os.chdir) - careful, this affects the whole process
os.chdir(SANDBOX)
print("cwd after chdir:", os.getcwd())

# Environment variables
print("\nHOME env var:", os.environ.get("HOME"))
print("PATH env var (truncated):", os.environ.get("PATH", "")[:80], "...")

# Setting an environment variable (only affects this process and its children)
os.environ["MY_TUTORIAL_VAR"] = "hello"
print("MY_TUTORIAL_VAR:", os.environ["MY_TUTORIAL_VAR"])

# OS info
print("\nOS name (os.name):", os.name)         # 'posix', 'nt', etc.
print("Path separator (os.sep):", repr(os.sep))
print("Line separator (os.linesep):", repr(os.linesep))


cwd: /home/claude
cwd after chdir: /tmp/ospathlib_tutorial_b_lut3an

HOME env var: /root
PATH env var (truncated): /home/claude/.npm-global/bin:/home/claude/.local/bin:/root/.local/bin:/usr/local ...
MY_TUTORIAL_VAR: hello

OS name (os.name): posix
Path separator (os.sep): '/'
Line separator (os.linesep): '\n'


### 1.2 Creating and removing directories

In [4]:
# os.mkdir - creates a single directory, fails if parent doesn't exist or dir exists
os.mkdir("os_demo_dir")
print("Created 'os_demo_dir':", os.path.exists("os_demo_dir"))

# os.makedirs - creates nested directories; exist_ok avoids error if they exist
os.makedirs("os_demo_dir/nested/deep", exist_ok=True)
print("Created nested dirs:", os.path.exists("os_demo_dir/nested/deep"))

# Removing directories
os.rmdir("os_demo_dir/nested/deep")   # only removes an EMPTY directory
print("Removed leaf dir, still exists 'nested'?", os.path.exists("os_demo_dir/nested"))


Created 'os_demo_dir': True
Created nested dirs: True
Removed leaf dir, still exists 'nested'? True


### 1.3 `os.path` — string-based path manipulation

`os.path` is the pre-`pathlib` way to manipulate paths. It works on plain strings.


In [5]:
p = os.path.join("data", "raw", "file.csv")
print("Joined path:", p)

print("basename:", os.path.basename(p))     # 'file.csv'
print("dirname:", os.path.dirname(p))       # 'data/raw'
print("splitext:", os.path.splitext(p))     # ('data/raw/file', '.csv')
print("split:", os.path.split(p))           # ('data/raw', 'file.csv')

print("\nabspath:", os.path.abspath(p))
print("isabs:", os.path.isabs(p))

print("\nexpanduser('~/x'):", os.path.expanduser("~/x"))
print("expandvars('$HOME/x'):", os.path.expandvars("$HOME/x"))


Joined path: data/raw/file.csv
basename: file.csv
dirname: data/raw
splitext: ('data/raw/file', '.csv')
split: ('data/raw', 'file.csv')

abspath: /tmp/ospathlib_tutorial_b_lut3an/data/raw/file.csv
isabs: False

expanduser('~/x'): /root/x
expandvars('$HOME/x'): /root/x


### 1.4 Checking file/dir existence and type

In [6]:
# Create a sample file to inspect
with open("os_demo_dir/sample.txt", "w") as f:
    f.write("sample content\n")

path = "os_demo_dir/sample.txt"
print("exists:", os.path.exists(path))
print("isfile:", os.path.isfile(path))
print("isdir:", os.path.isdir(path))
print("islink:", os.path.islink(path))
print("getsize (bytes):", os.path.getsize(path))
print("getmtime (timestamp):", os.path.getmtime(path))


exists: True
isfile: True
isdir: False
islink: False
getsize (bytes): 15
getmtime (timestamp): 1785912927.164206


### 1.5 Listing directories: `os.listdir` and `os.walk`

In [7]:
# Set up a small tree to walk
os.makedirs("os_demo_dir/sub1", exist_ok=True)
os.makedirs("os_demo_dir/sub2", exist_ok=True)
open("os_demo_dir/sub1/a.txt", "w").close()
open("os_demo_dir/sub1/b.csv", "w").close()
open("os_demo_dir/sub2/c.csv", "w").close()

# os.listdir - non-recursive, just names in one directory
print("listdir('os_demo_dir'):", os.listdir("os_demo_dir"))

# os.walk - recursive; yields (dirpath, dirnames, filenames) for every directory in the tree
print("\nos.walk output:")
for dirpath, dirnames, filenames in os.walk("os_demo_dir"):
    print(f"  dir: {dirpath} | subdirs: {dirnames} | files: {filenames}")


listdir('os_demo_dir'): ['sub2', 'sample.txt', 'nested', 'sub1']

os.walk output:
  dir: os_demo_dir | subdirs: ['sub2', 'nested', 'sub1'] | files: ['sample.txt']
  dir: os_demo_dir/sub2 | subdirs: [] | files: ['c.csv']
  dir: os_demo_dir/nested | subdirs: [] | files: []
  dir: os_demo_dir/sub1 | subdirs: [] | files: ['b.csv', 'a.txt']


### 1.6 Renaming and removing files

In [8]:
os.rename("os_demo_dir/sample.txt", "os_demo_dir/renamed.txt")
print("Renamed, exists now:", os.path.exists("os_demo_dir/renamed.txt"))

os.remove("os_demo_dir/renamed.txt")   # deletes a file (like os.unlink)
print("Removed, exists now:", os.path.exists("os_demo_dir/renamed.txt"))


Renamed, exists now: True
Removed, exists now: False


### 1.7 Process and system utilities

These have no `pathlib` equivalent — they stay in `os`.


In [9]:
print("Process ID:", os.getpid())
print("Parent process ID:", os.getppid())

# CPU count (also available via os.cpu_count())
print("CPU count:", os.cpu_count())

# Running a shell command (prefer subprocess module for anything serious,
# but os.system exists for quick one-offs)
exit_code = os.system("echo 'hello from a subprocess' > /dev/null")
print("Exit code from os.system:", exit_code)


Process ID: 523
Parent process ID: 522
CPU count: 1
Exit code from os.system: 0


---
# Part 2: The `pathlib` Module

`pathlib` represents filesystem paths as objects (`Path`) instead of plain strings, with methods attached directly to them. This tends to produce more readable and less error-prone code than `os.path`.


### 2.1 Creating `Path` objects

In [10]:
p1 = Path("data/file.txt")             # relative path
p2 = Path.cwd()                         # current working directory
p3 = Path.home()                        # user home directory
p4 = Path("/etc") / "hosts"             # absolute path via the / operator

print("p1:", p1)
print("p2 (cwd):", p2)
print("p3 (home):", p3)
print("p4:", p4)
print("type:", type(p1))


p1: data/file.txt
p2 (cwd): /tmp/ospathlib_tutorial_b_lut3an
p3 (home): /root
p4: /etc/hosts
type: <class 'pathlib.PosixPath'>


### 2.2 Joining paths with `/`

In [11]:
base = Path("data")
p = base / "raw" / "2024" / "file.csv"
print(p)

# Equivalent to os.path.join("data", "raw", "2024", "file.csv") but more readable


data/raw/2024/file.csv


### 2.3 Inspecting path components

In [12]:
p = Path("data/raw/2024/file.tar.gz")

print("name:   ", p.name)          # 'file.tar.gz'
print("stem:   ", p.stem)          # 'file.tar'  (only strips the LAST suffix)
print("suffix: ", p.suffix)        # '.gz'
print("suffixes:", p.suffixes)     # ['.tar', '.gz']
print("parent: ", p.parent)        # 'data/raw/2024'
print("parents:", list(p.parents)) # all ancestor directories
print("parts:  ", p.parts)         # tuple of all path components
print("anchor: ", p.anchor)        # root/drive part, empty for relative paths


name:    file.tar.gz
stem:    file.tar
suffix:  .gz
suffixes: ['.tar', '.gz']
parent:  data/raw/2024
parents: [PosixPath('data/raw/2024'), PosixPath('data/raw'), PosixPath('data'), PosixPath('.')]
parts:   ('data', 'raw', '2024', 'file.tar.gz')
anchor:  


### 2.4 Absolute paths and resolving

In [13]:
rel = Path("os_demo_dir") / ".." / "os_demo_dir" / "sub1"
print("original:      ", rel)
print("resolve():     ", rel.resolve())   # absolute, symlinks resolved, '..' collapsed
print("is_absolute(): ", rel.is_absolute())
print("absolute():    ", rel.absolute())  # absolute but does NOT collapse '..'


original:       os_demo_dir/../os_demo_dir/sub1
resolve():      /tmp/ospathlib_tutorial_b_lut3an/os_demo_dir/sub1
is_absolute():  False
absolute():     /tmp/ospathlib_tutorial_b_lut3an/os_demo_dir/../os_demo_dir/sub1


### 2.5 Creating and removing directories

In [14]:
demo = Path("pathlib_demo")

demo.mkdir(exist_ok=True)                              # single dir
(demo / "nested" / "deep").mkdir(parents=True, exist_ok=True)  # nested dirs

print("Created:", (demo / "nested" / "deep").exists())

(demo / "nested" / "deep").rmdir()   # removes empty dir only
print("Removed leaf, 'nested' still exists:", (demo / "nested").exists())


Created: True
Removed leaf, 'nested' still exists: True


### 2.6 Checking existence and type

In [15]:
f = demo / "sample.txt"
f.write_text("hello from pathlib\n")   # create + write in one call

print("exists():   ", f.exists())
print("is_file():  ", f.is_file())
print("is_dir():   ", f.is_dir())
print("is_symlink():", f.is_symlink())


exists():    True
is_file():   True
is_dir():    False
is_symlink(): False


### 2.7 Reading and writing files directly

In [16]:
f = demo / "notes.txt"

# Text
f.write_text("line one\nline two\n")
print("read_text():\n", f.read_text())

# Bytes
bf = demo / "data.bin"
bf.write_bytes(b"\x00\x01\x02")
print("read_bytes():", bf.read_bytes())

# Path objects also work directly with the built-in open()
with f.open("a") as fh:   # 'a' = append mode
    fh.write("line three\n")
print("\nAfter append:\n", f.read_text())


read_text():
 line one
line two

read_bytes(): b'\x00\x01\x02'

After append:
 line one
line two
line three



### 2.8 Listing contents: `iterdir`, `glob`, `rglob`

In [17]:
# Build a small tree
(demo / "sub1").mkdir(exist_ok=True)
(demo / "sub2").mkdir(exist_ok=True)
(demo / "sub1" / "a.txt").write_text("a")
(demo / "sub1" / "b.csv").write_text("b")
(demo / "sub2" / "c.csv").write_text("c")

# iterdir - non-recursive, immediate children only
print("iterdir():")
for item in demo.iterdir():
    print(" ", item)

# glob - pattern matching, non-recursive by default
print("\nglob('*.txt') in demo (none, they're in subdirs):", list(demo.glob("*.txt")))
print("glob('sub1/*.csv'):", list(demo.glob("sub1/*.csv")))

# rglob - recursive glob, searches all subdirectories
print("\nrglob('*.csv') (recursive):")
for item in demo.rglob("*.csv"):
    print(" ", item)


iterdir():
  pathlib_demo/sub2
  pathlib_demo/data.bin
  pathlib_demo/sample.txt
  pathlib_demo/nested
  pathlib_demo/sub1
  pathlib_demo/notes.txt

glob('*.txt') in demo (none, they're in subdirs): [PosixPath('pathlib_demo/sample.txt'), PosixPath('pathlib_demo/notes.txt')]
glob('sub1/*.csv'): [PosixPath('pathlib_demo/sub1/b.csv')]

rglob('*.csv') (recursive):
  pathlib_demo/sub2/c.csv
  pathlib_demo/sub1/b.csv


### 2.9 Renaming, moving, and deleting

In [18]:
old = demo / "notes.txt"
new = demo / "notes_renamed.txt"

old.rename(new)
print("Renamed, new exists:", new.exists(), "| old exists:", old.exists())

new.unlink()                    # delete a file
print("Deleted, exists now:", new.exists())

# unlink(missing_ok=True) avoids raising if the file is already gone (Python 3.8+)
new.unlink(missing_ok=True)     # no error even though it's already deleted
print("Safe unlink on missing file: no exception raised")


Renamed, new exists: True | old exists: False
Deleted, exists now: False
Safe unlink on missing file: no exception raised


### 2.10 File metadata with `.stat()`

In [19]:
f = demo / "sub1" / "a.txt"
st = f.stat()

print("Size (bytes):     ", st.st_size)
print("Last modified:    ", st.st_mtime)
print("Mode (permissions):", oct(st.st_mode))


Size (bytes):      1
Last modified:     1785912927.2371445
Mode (permissions): 0o100644


### 2.11 `pathlib` vs `os.path` — quick comparison

In [20]:
p_str = "data/raw/file.csv"
p_path = Path(p_str)

print("os.path.basename:", os.path.basename(p_str), " | pathlib .name:  ", p_path.name)
print("os.path.dirname: ", os.path.dirname(p_str), " | pathlib .parent:", p_path.parent)
print("os.path.join:    ", os.path.join("data", "file.csv"), "     | pathlib /:      ", Path("data") / "file.csv")
print("os.path.exists:  ", os.path.exists(p_str), "         | pathlib .exists():", p_path.exists())


os.path.basename: file.csv  | pathlib .name:   file.csv
os.path.dirname:  data/raw  | pathlib .parent: data/raw
os.path.join:     data/file.csv      | pathlib /:       data/file.csv
os.path.exists:   False          | pathlib .exists(): False


---
# Part 3: The `sys` Module

`sys` provides access to variables and functions that interact directly with the Python interpreter — not the OS filesystem, but the runtime itself.


### 3.1 Interpreter info

In [21]:
print("sys.version:      ", sys.version)
print("sys.version_info: ", sys.version_info)
print("sys.platform:     ", sys.platform)     # 'linux', 'darwin', 'win32', etc.
print("sys.executable:   ", sys.executable)   # path to the Python binary running this
print("sys.prefix:       ", sys.prefix)       # install prefix


sys.version:       3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
sys.version_info:  sys.version_info(major=3, minor=12, micro=3, releaselevel='final', serial=0)
sys.platform:      linux
sys.executable:    /usr/bin/python3
sys.prefix:        /usr


### 3.2 Command-line arguments (`sys.argv`)

In [22]:
# In a script run as `python myscript.py arg1 arg2`, sys.argv would be:
#   ['myscript.py', 'arg1', 'arg2']
# In a notebook, sys.argv reflects how the Jupyter kernel itself was launched.
print("sys.argv:", sys.argv)


sys.argv: ['/usr/local/lib/python3.12/dist-packages/ipykernel_launcher.py', '-f', '/tmp/tmpeb3bjs12.json', '--HistoryManager.hist_file=:memory:']


### 3.3 The module search path (`sys.path`)

In [23]:
# sys.path is the list of directories Python searches when you do `import something`.
# It's how Python finds modules and packages.
for entry in sys.path:
    print(entry)


/usr/lib/python312.zip
/usr/lib/python3.12
/usr/lib/python3.12/lib-dynload

/usr/local/lib/python3.12/dist-packages
/usr/lib/python3/dist-packages


In [24]:
# You can add a directory to sys.path at runtime to make local modules importable
# (a common pattern before proper packaging/venvs)
extra_dir = str(SANDBOX / "my_local_modules")
os.makedirs(extra_dir, exist_ok=True)
sys.path.insert(0, extra_dir)
print("Added to sys.path:", extra_dir in sys.path)
sys.path.remove(extra_dir)  # clean up


Added to sys.path: True


### 3.4 Loaded modules (`sys.modules`)

In [25]:
# sys.modules is a cache of every module that's been imported in this process
print("Is 'os' loaded?", "os" in sys.modules)
print("Is 'json' loaded (before import)?", "json" in sys.modules)
import json
print("Is 'json' loaded (after import)?", "json" in sys.modules)


Is 'os' loaded? True
Is 'json' loaded (before import)? True
Is 'json' loaded (after import)? True


### 3.5 Memory: object size and reference counting

In [26]:
x = [1, 2, 3, 4, 5]
print("Size of list in bytes:", sys.getsizeof(x))
print("Size of int 42 in bytes:", sys.getsizeof(42))
print("Size of empty string:", sys.getsizeof(""))

# Reference count for an object (how many references point to it)
print("\nRefcount for x:", sys.getrefcount(x))  # includes the temporary ref from the function call itself


Size of list in bytes: 104
Size of int 42 in bytes: 28
Size of empty string: 41

Refcount for x: 2


### 3.6 stdout, stderr, and redirecting output

In [27]:
# sys.stdout / sys.stderr are file-like objects. print() writes to sys.stdout by default.
sys.stdout.write("Written directly via sys.stdout.write()\n")

# Redirecting stdout temporarily (e.g. to capture print output into a string)
import io
buffer = io.StringIO()
old_stdout = sys.stdout
sys.stdout = buffer
print("this goes into the buffer, not the visible output")
sys.stdout = old_stdout   # restore

print("Captured from buffer:", buffer.getvalue().strip())


Written directly via sys.stdout.write()
Captured from buffer: this goes into the buffer, not the visible output


### 3.7 Exiting a program with `sys.exit`

In [28]:
# sys.exit(code) raises SystemExit, which normally terminates the interpreter.
# In a script: sys.exit(0) means success, non-zero means error.
# We won't actually call it here since it would stop the notebook kernel's execution
# of this cell (Jupyter catches SystemExit gracefully, but it's disruptive) —
# shown as reference only:

example_code = '''
import sys

if len(sys.argv) < 2:
    print("Usage: script.py <argument>")
    sys.exit(1)   # non-zero = error exit code
'''
print(example_code)



import sys

if len(sys.argv) < 2:
    print("Usage: script.py <argument>")
    sys.exit(1)   # non-zero = error exit code



### 3.8 Recursion limit

In [29]:
print("Current recursion limit:", sys.getrecursionlimit())
# sys.setrecursionlimit(3000)  # can be raised, but doing so risks a C stack overflow


Current recursion limit: 1000


---
# Part 4: Combined Example

A small utility that uses all three modules together: `sys` for command-line-style arguments, `pathlib` for filesystem traversal, and `os` for environment lookups.


In [30]:
def find_large_files(directory, min_size_kb=1, pattern="*"):
    """
    Recursively find files in `directory` matching `pattern` that are
    at least `min_size_kb` kilobytes.
    Demonstrates pathlib for traversal + os for env var fallback + sys for reporting.
    """
    base = Path(directory)
    if not base.exists():
        print(f"Directory not found: {base}", file=sys.stderr)
        return []

    results = []
    for file_path in base.rglob(pattern):
        if file_path.is_file():
            size_kb = file_path.stat().st_size / 1024
            if size_kb >= min_size_kb:
                results.append((str(file_path), round(size_kb, 3)))
    return results


# Use an env var for a default search root, falling back to the sandbox
search_root = os.environ.get("TUTORIAL_SEARCH_DIR", str(demo))

print(f"Searching under: {search_root}")
print(f"Python executable: {sys.executable}\n")

for name, size in find_large_files(search_root, min_size_kb=0):
    print(f"  {name}  ({size} KB)")


Searching under: pathlib_demo
Python executable: /usr/bin/python3

  pathlib_demo/data.bin  (0.003 KB)
  pathlib_demo/sample.txt  (0.019 KB)
  pathlib_demo/sub2/c.csv  (0.001 KB)
  pathlib_demo/sub1/b.csv  (0.001 KB)
  pathlib_demo/sub1/a.txt  (0.001 KB)


### 4.1 Cleanup

Remove the sandbox directory and everything created in it.


In [31]:
import shutil

os.chdir(Path.home())  # move out of the sandbox before deleting it
shutil.rmtree(SANDBOX, ignore_errors=True)
print("Sandbox removed:", not SANDBOX.exists())


Sandbox removed: True


---
## Summary

| Module | Best for |
|---|---|
| `os` | Environment variables, process info (`getpid`, `system`), low-level OS calls |
| `os.path` | String-based path manipulation (legacy codebases, quick scripts) |
| `pathlib` | Object-oriented path manipulation, reading/writing files, recursive search (`rglob`) — preferred for new code |
| `sys` | Interpreter internals: `argv`, `path`, `modules`, `stdout`/`stderr`, `exit`, memory/recursion introspection |

**Rule of thumb:** reach for `pathlib` first for anything involving paths and files, `os`/`os.environ` for environment and process-level operations, and `sys` when you need to interact with the interpreter itself (arguments, import machinery, exit codes, stdio streams).
